# 1. Introduction

This notebook performs the linguistic preprocessing and feature extraction procedures used in the study. Raw textual responses are cleaned, tokenized, and analyzed to generate a set of linguistic indicators associated with persuasive communication.

The extracted features include measures related to argument production, connective use, polite language, rude language, powerless language, and other lexical characteristics. These variables are subsequently used in the following stages of the project, including principal component analysis (PCA) and cluster analysis.

The output of this notebook is a structured dataset containing one row per participant and the linguistic variables required for the subsequent analyses.

In [47]:
# Imports
import pandas as pd
import nltk
import string

from unidecode import unidecode

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import bigrams, trigrams

## 2. Data Loading

In [48]:
# Read the data locally
#path =r'C:\Users\juanm\Jupyter nootebooks\Linkedin\Proyecto celina\Linkedin\PersuasionPART1_rawdata_token.csv'
path =r'data\PersuasionPART1_rawdata_token.csv'

# Read the data from git hub
#df = pd.read_csv("/data/persuasion_raw_data.csv")

# Load data
df = pd.read_csv(path,sep=',', encoding='utf-8')

In [49]:
df.head(3)

,Est,Diada,Edad,Género,ChatGPT_uso,ChatGPT_cal,Justificacion_1,Posicion,ChatGPT_cal_2,Justificacion_2,...,NEO_6,NEO_7,NEO_8,NEO_9,NEO_10,NEO_11,NEO_12,Impuso,Cant_arg,Extro
0,1,3,23,Mujer,2,3,Que en ciertas ocasiones nos brinda informació...,En contra,3,"Respecto al ChatGPT, estoy a favor porque para...",...,2,3,2,2,2,2,3,0,7,31
1,2,3,35,Varón,2,3,Considero que siempre y cuando sea complementa...,A favor,3,Creo que es necesario tener en cuenta que es u...,...,1,3,3,2,3,3,3,1,10,35
2,3,4,19,Varón,2,3,"Es una herramienta facilitadora, pero no un re...",A favor,2,"Después del debate, entiendo que un buen uso a...",...,1,3,2,3,2,3,3,0,7,28


## 3. Text Preprocessing

In [50]:
# Text preprocessing

# The original transcripts contained POS-tagging labels added
# during a previous annotation stage. These labels must be removed
# before tokenization and feature extraction.

# Standard preprocessing includes:
# - Accent normalization
# - Punctuation removal
# - Tokenization
# - Removal of custom annotation tags

In [51]:
custom_stop_words = ['arg', 'carg', 'ad', 'ins', 'pr', 'pv', '*', '()']


def preprocess_text(text):
    # Eliminar tildes
    text = unidecode(text)
    text = ''.join([char if char not in string.punctuation else ' ' for char in text])
    # Tokenizar el texto
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word not in custom_stop_words]
    return tokens

df['Int_toke'] = df['Interaccion_dialogico'].apply(preprocess_text)

print(df['Int_toke'])

0     [bueno, yo, entiendo, tu, postura, pero, yo, e...
1     [bueno, teniendo, en, cuenta, lo, que, es, el,...
2     [bueno, estoy, a, favor, bueno, estoy, a, favo...
3     [claro, pero, si, en, cuestion, academica, vos...
4     [bueno, yo, recomendaria, eh, el, chatgpt, en,...
5     [por, que, recomendarias, el, uso, de, chatgpt...
6     [ok, bueno, yo, considero, que, el, chat, gpt,...
7     [gracias, yo, en, teoria, estoy, de, favor, en...
8     [bueno, a, ver, yo, estoy, a, favor, porque, s...
9     [queres, empezar, vos, la, verdad, creo, que, ...
10    [bueno, eh, que, opinas, que, es, claro, esta,...
11    [queres, empezar, vos, hasta, ahi, llegamos, j...
12    [arrancar, arranco, yo, bueno, arranco, justo,...
13    [como, quieras, que, no, es, tan, fiable, y, b...
14    [bueno, yo, estoy, a, favor, del, uso, de, la,...
15    [de, todas, formas, me, parece, que, es, muy, ...
16    [claro, pero, justamente, no, te, convendria, ...
17    [bueno, a, mi, me, parece, eh, que, estoy,

In [52]:
def count_tokens_row(row):
    return len(row['Int_toke'])

df['Total_tokens'] = df.apply(count_tokens_row, axis=1)

print(df[['Int_toke','Total_tokens']])

                                             Int_toke  Total_tokens
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...           457
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...          1033
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...           524
3   [claro, pero, si, en, cuestion, academica, vos...           615
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...            90
5   [por, que, recomendarias, el, uso, de, chatgpt...           183
6   [ok, bueno, yo, considero, que, el, chat, gpt,...           364
7   [gracias, yo, en, teoria, estoy, de, favor, en...           350
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...          1356
9   [queres, empezar, vos, la, verdad, creo, que, ...           926
10  [bueno, eh, que, opinas, que, es, claro, esta,...          2002
11  [queres, empezar, vos, hasta, ahi, llegamos, j...           808
12  [arrancar, arranco, yo, bueno, arranco, justo,...           352
13  [como, quieras, que, no, es, tan, fiable, y,

### 4. Connector Extraction

In [53]:
connectors_words = [
    'y', 'porque', 'pero', 'o', 'eventualmente', 'recientemente', 
    'mientras', 'adicionalmente', 'ademas', 'tambien', 'segundo', 
    'primero', 'tercero', 'aunque', 'asi', 'luego', 'pues', 'entonces',
    'quizas'
]

connectors_bigrams = [
    ('sin', 'embargo'), ('contrario', 'de'), ('en', 'consecuencia'),
    ('a', 'menos'), ('a', 'pesar'), ('aun', 'si'), ('lo', 'último'),
    ('por', 'último'), ('por', 'ejemplo'), ('en', 'síntesis'), 
    ('para', 'concluir'), ('en', 'conclusión'), ('según', 'esto'),
    ('por', 'consiguiente'), ('en', 'cambio'), ('en', 'efecto'), 
    ('por', 'ello'), ('en', 'fin'), ('de', 'hecho'), ('a', 'continuación'),
    ('a', 'propósito'), ('al', 'contrario'), ('en', 'cualquier'), ('con', 'todo'),
    ('sin', 'duda'), ('por', 'ende'), ('por', 'eso'), ('no', 'obstante')
    
]

connectors_trigrams = [
    ('a', 'diferencia', 'de'), ('en', 'este', 'sentido'), ('por', 'lo', 'tanto'), 
    ('de', 'este', 'modo'), ('de', 'otro', 'modo'), ('primero', 'que', 'todo'), 
    ('en', 'otras', 'palabras'), ('o', 'mejor', 'dicho'), ('todo', 'en', 'todo'), 
    ('para', 'esta', 'posición'), ('de', 'acuerdo', 'con'), ('en', 'primer', 'lugar'), 
    ('en', 'segundo', 'lugar'), ('en', 'tercer', 'lugar'), ('por', 'esta', 'razón'),
    ('por', 'otra', 'parte'), ('como', 'resultado', 'de'), ('debido', 'a', 'esto'),
    ('por', 'tal', 'motivo'), ('con', 'base', 'en')
]

In [54]:
def count_connectors_words(row):
    frec_row = 0
    for word in connectors_words:
        frec_row += row['Int_toke'].count(word)
    return frec_row

df['connectors_total_words'] = df.apply(count_connectors_words, axis=1)

print(df[['Int_toke', 'connectors_total_words']])

                                             Int_toke  connectors_total_words
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...                      24
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...                      72
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...                      38
3   [claro, pero, si, en, cuestion, academica, vos...                      53
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...                       7
5   [por, que, recomendarias, el, uso, de, chatgpt...                       8
6   [ok, bueno, yo, considero, que, el, chat, gpt,...                      32
7   [gracias, yo, en, teoria, estoy, de, favor, en...                      22
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...                     117
9   [queres, empezar, vos, la, verdad, creo, que, ...                      70
10  [bueno, eh, que, opinas, que, es, claro, esta,...                     145
11  [queres, empezar, vos, hasta, ahi, llegamos, j...           

In [55]:
def count_connectors_bigrams(row):
    frec_row = 0
    tokens = row['Int_toke']
    
    tokens_lower = [token.lower() for token in tokens]
    
    tokens_bigramas = list(bigrams(tokens_lower))
    
    for bigram in tokens_bigramas:
        if bigram in connectors_bigrams:
            frec_row += 1
    
    return frec_row

df['connectors_total_bigrams'] = df.apply(count_connectors_bigrams, axis=1)

print(df[['Int_toke', 'connectors_total_bigrams']])

                                             Int_toke  \
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...   
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...   
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...   
3   [claro, pero, si, en, cuestion, academica, vos...   
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...   
5   [por, que, recomendarias, el, uso, de, chatgpt...   
6   [ok, bueno, yo, considero, que, el, chat, gpt,...   
7   [gracias, yo, en, teoria, estoy, de, favor, en...   
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...   
9   [queres, empezar, vos, la, verdad, creo, que, ...   
10  [bueno, eh, que, opinas, que, es, claro, esta,...   
11  [queres, empezar, vos, hasta, ahi, llegamos, j...   
12  [arrancar, arranco, yo, bueno, arranco, justo,...   
13  [como, quieras, que, no, es, tan, fiable, y, b...   
14  [bueno, yo, estoy, a, favor, del, uso, de, la,...   
15  [de, todas, formas, me, parece, que, es, muy, ...   
16  [claro, pero, justamente, n

In [56]:
def count_connectors_trigrams(row):
    frec_row = 0
    tokens = row['Int_toke']

    tokens_lower = [token.lower() for token in tokens]
    
    tokens_trigrams = list(trigrams(tokens_lower))
    
    for trigram in tokens_trigrams:
        if trigram in connectors_trigrams:
            frec_row += 1
    
    return frec_row

df['connectors_total_trigrams'] = df.apply(count_connectors_trigrams, axis=1)

print(df[['Int_toke', 'connectors_total_trigrams']])

                                             Int_toke  \
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...   
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...   
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...   
3   [claro, pero, si, en, cuestion, academica, vos...   
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...   
5   [por, que, recomendarias, el, uso, de, chatgpt...   
6   [ok, bueno, yo, considero, que, el, chat, gpt,...   
7   [gracias, yo, en, teoria, estoy, de, favor, en...   
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...   
9   [queres, empezar, vos, la, verdad, creo, que, ...   
10  [bueno, eh, que, opinas, que, es, claro, esta,...   
11  [queres, empezar, vos, hasta, ahi, llegamos, j...   
12  [arrancar, arranco, yo, bueno, arranco, justo,...   
13  [como, quieras, que, no, es, tan, fiable, y, b...   
14  [bueno, yo, estoy, a, favor, del, uso, de, la,...   
15  [de, todas, formas, me, parece, que, es, muy, ...   
16  [claro, pero, justamente, n

### 5. Politeness  Indicators

In [57]:
politeness_words = [
    'quisiera', 'quisieras','podrias','podrias', 'gracias', 'podria', 'podriamos',
    'disculpa', 'perdon', 'porfa', 'amable', 'gustaria', 'permiso',
    'esperemos','favor','haria','creo','permiso','agradezco',
]

politeness_bigrams = [('para','mi'), ('me','parece'), ('mi','parecer'),
    ('te', 'importaria'), ('seria', 'posible'),
]

politeness_trigrams = [('te', 'parece', 'si'), ('que', 'tal', 'si'), ('yo', 'lo', 'veo')]

In [58]:
def count_words_politeness (row):
    frec_row = 0
    for word in politeness_words:
        frec_row += row['Int_toke'].count(word)
    return frec_row

df['politeness_words'] = df.apply(count_words_politeness, axis=1)

print(df[['Int_toke', 'politeness_words']])

                                             Int_toke  politeness_words
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...                 1
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...                 6
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...                 3
3   [claro, pero, si, en, cuestion, academica, vos...                 2
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...                 0
5   [por, que, recomendarias, el, uso, de, chatgpt...                 9
6   [ok, bueno, yo, considero, que, el, chat, gpt,...                 2
7   [gracias, yo, en, teoria, estoy, de, favor, en...                 3
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...                 2
9   [queres, empezar, vos, la, verdad, creo, que, ...                 4
10  [bueno, eh, que, opinas, que, es, claro, esta,...                 9
11  [queres, empezar, vos, hasta, ahi, llegamos, j...                14
12  [arrancar, arranco, yo, bueno, arranco, justo,...           

In [59]:
def count_bigrams_politeness(row):
    frec_row = 0
    tokens = row['Int_toke']
    
    tokens_lower = [token.lower() for token in tokens]
    
    tokens_bigramas = list(bigrams(tokens_lower))
    
    for bigram in tokens_bigramas:
        if bigram in politeness_bigrams:
            frec_row += 1
    
    return frec_row

df['politeness_bigrams'] = df.apply(count_bigrams_politeness, axis=1)

print(df[['Int_toke', 'politeness_bigrams']])

                                             Int_toke  politeness_bigrams
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...                   5
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...                   6
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...                   3
3   [claro, pero, si, en, cuestion, academica, vos...                   0
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...                   1
5   [por, que, recomendarias, el, uso, de, chatgpt...                   1
6   [ok, bueno, yo, considero, que, el, chat, gpt,...                   2
7   [gracias, yo, en, teoria, estoy, de, favor, en...                   0
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...                   3
9   [queres, empezar, vos, la, verdad, creo, que, ...                   0
10  [bueno, eh, que, opinas, que, es, claro, esta,...                   9
11  [queres, empezar, vos, hasta, ahi, llegamos, j...                   9
12  [arrancar, arranco, yo, bueno, arr

In [60]:
def count_trigrams_politeness(row):
    frec_row = 0
    tokens = row['Int_toke']
    
    tokens_lower = [token.lower() for token in tokens]
    
    tokens_trigrams = list(trigrams(tokens_lower))
    
    for trigram in tokens_trigrams:
        if trigram in politeness_trigrams:
            frec_row  += 1
    
    return frec_row

df['politeness_trigrams'] = df.apply(count_trigrams_politeness, axis=1)

print(df[['Int_toke', 'politeness_trigrams']])

                                             Int_toke  politeness_trigrams
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...                    0
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...                    0
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...                    0
3   [claro, pero, si, en, cuestion, academica, vos...                    0
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...                    0
5   [por, que, recomendarias, el, uso, de, chatgpt...                    0
6   [ok, bueno, yo, considero, que, el, chat, gpt,...                    0
7   [gracias, yo, en, teoria, estoy, de, favor, en...                    0
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...                    0
9   [queres, empezar, vos, la, verdad, creo, que, ...                    0
10  [bueno, eh, que, opinas, que, es, claro, esta,...                    0
11  [queres, empezar, vos, hasta, ahi, llegamos, j...                    0
12  [arrancar, arranco, y

### 6. Rudeness Indicators

In [61]:
rud_words = [
    'deberias', 'debes', 'deber', 'debemos', 'tenemos', 'obligas', 'forzado', 'impones', 
    'tienes', 'necesitas', 'insistes', 'manda', 'mandas', 'prohibido', 'mal', 
    'exigido', 'exigir', 'necesito', 'obligatorio', 'no', 'nunca', 'jamas', 
    'imposible'
]

rud_bigrams = [
    ('tienes', 'que'), ('hay', 'que'), ('se', 'debe'), ('es', 'imposible'), 
    ('no', 'puedes'), ('no', 'debes'), ('te', 'exijo'), ('no', 'es'), 
    ('no', 'quiero'), ('hazlo', 'ahora'), ('no', 'hagas'), ('lo', 'peor'), 
    ('es', 'obligatorio'), ('no', 'vale'), ('es', 'urgente'), ('no', 'importa'), 
    ('te', 'obligo'), ('es', 'necesario'), ('te', 'mando'), ('sin', 'permiso'), 
    ('no', 'acepto'), ('nada', 'de'), ('necesito', 'que'), ('hazlo', 'ya'), 
    ('no', 'aceptes'), ('es','asi')
]


rud_trigrams = [
    ('la', 'unica', 'manera'), ('no', 'hay', 'duda'), ('no', 'es', 'posible'), 
    ('no', 'hay', 'tiempo'), ('no', 'lo', 'hare'), ('no', 'me', 'importa'), 
    ('no', 'lo', 'se'), ('no', 'quiero', 'saber'), ('no', 'es', 'necesario'), 
    ('no', 'vas', 'a'), ('poder',), ('te', 'lo', 'exijo'), ('haz', 'lo', 'que'), 
    ('sea', 'que'), ('te', 'lo', 'dije'), ('tienes', 'que', 'hacerlo'), 
    ('no', 'me', 'importas'), ('no', 'hay', 'excusas'), ('no', 'tienes', 'derecho'),
    ('no', 'tienes', 'elección'), ('te', 'voy', 'a'),
]

In [62]:
def count_word_rud(row):
    frec_row = 0
    for word in rud_words:
        frec_row += row['Int_toke'].count(word)
    return frec_row

df['rud_words'] = df.apply(count_word_rud, axis=1)

print(df[['Int_toke', 'rud_words']])

                                             Int_toke  rud_words
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...         14
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...         35
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...         15
3   [claro, pero, si, en, cuestion, academica, vos...         13
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...          1
5   [por, que, recomendarias, el, uso, de, chatgpt...          3
6   [ok, bueno, yo, considero, que, el, chat, gpt,...         13
7   [gracias, yo, en, teoria, estoy, de, favor, en...         12
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...         35
9   [queres, empezar, vos, la, verdad, creo, que, ...         38
10  [bueno, eh, que, opinas, que, es, claro, esta,...         81
11  [queres, empezar, vos, hasta, ahi, llegamos, j...         29
12  [arrancar, arranco, yo, bueno, arranco, justo,...         13
13  [como, quieras, que, no, es, tan, fiable, y, b...          5
14  [bueno, yo, estoy, a,

In [63]:
def count_bigrams_rud(row):
    frec_row = 0
    tokens = row['Int_toke']
    
    tokens_lower = [token.lower() for token in tokens]
    
    tokens_bigrams = list(bigrams(tokens_lower))
    
    for bigram in tokens_bigrams:
        if bigram in rud_bigrams:
            frec_row += 1
    
    return frec_row

df['rud_bi'] = df.apply(count_bigrams_rud, axis=1)

print(df[['Int_toke', 'rud_bi']])

                                             Int_toke  rud_bi
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...       0
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...       4
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...       3
3   [claro, pero, si, en, cuestion, academica, vos...       1
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...       0
5   [por, que, recomendarias, el, uso, de, chatgpt...       0
6   [ok, bueno, yo, considero, que, el, chat, gpt,...       3
7   [gracias, yo, en, teoria, estoy, de, favor, en...       4
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...       2
9   [queres, empezar, vos, la, verdad, creo, que, ...       5
10  [bueno, eh, que, opinas, que, es, claro, esta,...      10
11  [queres, empezar, vos, hasta, ahi, llegamos, j...       2
12  [arrancar, arranco, yo, bueno, arranco, justo,...       2
13  [como, quieras, que, no, es, tan, fiable, y, b...       1
14  [bueno, yo, estoy, a, favor, del, uso, de, la,...       1
15  [de,

In [64]:
def count_trigrams_rud(row):
    frec_row = 0
    tokens = row['Int_toke']
    
    tokens_lower = [token.lower() for token in tokens]
    
    tokens_trigrams = list(trigrams(tokens_lower))
    
    for trigram in tokens_trigrams:
        if trigram in rud_trigrams:
            frec_row += 1
    
    return frec_row

df['rud_tri'] = df.apply(count_trigrams_rud, axis=1)

print(df[['Int_toke', 'rud_tri']])

                                             Int_toke  rud_tri
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...        0
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...        0
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...        0
3   [claro, pero, si, en, cuestion, academica, vos...        0
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...        0
5   [por, que, recomendarias, el, uso, de, chatgpt...        0
6   [ok, bueno, yo, considero, que, el, chat, gpt,...        0
7   [gracias, yo, en, teoria, estoy, de, favor, en...        0
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...        1
9   [queres, empezar, vos, la, verdad, creo, que, ...        0
10  [bueno, eh, que, opinas, que, es, claro, esta,...        2
11  [queres, empezar, vos, hasta, ahi, llegamos, j...        0
12  [arrancar, arranco, yo, bueno, arranco, justo,...        0
13  [como, quieras, que, no, es, tan, fiable, y, b...        0
14  [bueno, yo, estoy, a, favor, del, uso, de, la,...  

### 7. Powerless Language Indicators

In [65]:
powerless_words = ['eh', 'mhm','digamos']

In [66]:
def count_powerless_words(row):
    frec_row = 0
    for word in powerless_words:
        frec_row += row['Int_toke'].count(word)
    return frec_row

df['powerless_total'] = df.apply(count_powerless_words, axis=1)

print(df[['Int_toke', 'powerless_total']])

                                             Int_toke  powerless_total
0   [bueno, yo, entiendo, tu, postura, pero, yo, e...                0
1   [bueno, teniendo, en, cuenta, lo, que, es, el,...               12
2   [bueno, estoy, a, favor, bueno, estoy, a, favo...                4
3   [claro, pero, si, en, cuestion, academica, vos...                4
4   [bueno, yo, recomendaria, eh, el, chatgpt, en,...                4
5   [por, que, recomendarias, el, uso, de, chatgpt...                3
6   [ok, bueno, yo, considero, que, el, chat, gpt,...               10
7   [gracias, yo, en, teoria, estoy, de, favor, en...                1
8   [bueno, a, ver, yo, estoy, a, favor, porque, s...               12
9   [queres, empezar, vos, la, verdad, creo, que, ...                3
10  [bueno, eh, que, opinas, que, es, claro, esta,...                9
11  [queres, empezar, vos, hasta, ahi, llegamos, j...                0
12  [arrancar, arranco, yo, bueno, arranco, justo,...                5
13  [c

Once we have all the frecuencies we need we could delete the columns that we wont going to use anymore. Is more easy to just make a new df with all the variables we need

### 8. Dataset Construction

In [67]:
# Calculations

# connectors
df['connectors_total'] = df['connectors_total_words'] + df['connectors_total_bigrams'] + df['connectors_total_trigrams']

# politenessness
df['politeness_total'] = df['politeness_words'] + df['politeness_bigrams'] + df['politeness_trigrams']

# Rudeness
df['rud_total'] = df['rud_words'] + df['rud_bi'] + df['rud_tri']

# Df
df = df[['Est','Edad', 'Impuso', 'Extro', 'Cant_arg', 'connectors_total', 'politeness_total', 'rud_total', 'powerless_total']]

### 9. Export for PCA Analysis

In [68]:
# Save the data locally if necessary
df.to_csv(r'data\PersuasionPART2_numeric.csv', sep=",", index=False)